# Notebook pour entraîner un modèle LoRA avec Diffusers et un dataset TopDown

**Version adaptée pour environnement local avec GPU**

## Vérification de l'environnement et du GPU

In [1]:
# Cloner le repository GitHub (uniquement sur Colab)
try:
    from google.colab import drive
    import os
    import shutil
    
    # Vérifier si on est sur Colab
    IN_COLAB = True
    print("✓ Environnement Google Colab détecté")
    
    # Informations du repository
    GITHUB_REPO = "https://github.com/romainsebire/CADOTProject.git"
    BRANCH = "remy"
    TARGET_DIR = "/content/CADOTProject"
    
    # Vérifier si c'est un vrai repo git
    is_git_repo = os.path.exists(os.path.join(TARGET_DIR, ".git"))
    
    if os.path.exists(TARGET_DIR) and not is_git_repo:
        print(f"⚠️  {TARGET_DIR} existe mais n'est pas un dépôt Git")
        print(f"   Suppression du dossier...")
        shutil.rmtree(TARGET_DIR)
        print(f"✓ Dossier supprimé")
    
    # Cloner le repo si pas déjà présent
    if not os.path.exists(TARGET_DIR):
        print(f"\n📥 Clonage du repository depuis GitHub...")
        print(f"   Repo: {GITHUB_REPO}")
        print(f"   Branche: {BRANCH}")
        
        !git clone -b {BRANCH} {GITHUB_REPO} {TARGET_DIR}
        
        print(f"✓ Repository cloné avec succès dans {TARGET_DIR}")
    else:
        print(f"✓ Repository Git déjà présent dans {TARGET_DIR}")
        print(f"   Mise à jour du repository...")
        
        # Se déplacer dans le répertoire et pull les dernières modifications
        %cd {TARGET_DIR}
        !git checkout {BRANCH}
        !git pull origin {BRANCH}
        %cd /content
        
        print(f"✓ Repository mis à jour")
    
    # Vérifier le contenu
    print(f"\n📂 Contenu du projet:")
    !ls -la {TARGET_DIR}
    
    # Vérifier spécifiquement le dossier CADOT_Dataset
    dataset_path = f"{TARGET_DIR}/CADOT_Dataset"
    if os.path.exists(dataset_path):
        print(f"\n✓ Dossier CADOT_Dataset trouvé")
        !ls -l {dataset_path}
    else:
        print(f"\n⚠️  ATTENTION: Dossier CADOT_Dataset non trouvé!")
        print(f"   Le dataset n'est peut-être pas dans le repo GitHub.")
        print(f"   Vous devrez l'uploader manuellement ou utiliser Google Drive.")
    
except ImportError:
    IN_COLAB = False
    print("Non exécuté sur Colab - Utilisation de l'environnement local")

✓ Environnement Google Colab détecté

📥 Clonage du repository depuis GitHub...
   Repo: https://github.com/romainsebire/CADOTProject.git
   Branche: remy
Cloning into '/content/CADOTProject'...
fatal: could not read Username for 'https://github.com': No such device or address
fatal: could not read Username for 'https://github.com': No such device or address
✓ Repository cloné avec succès dans /content/CADOTProject

📂 Contenu du projet:
ls: cannot access '/content/CADOTProject': No such file or directory

⚠️  ATTENTION: Dossier CADOT_Dataset non trouvé!
   Le dataset n'est peut-être pas dans le repo GitHub.
   Vous devrez l'uploader manuellement ou utiliser Google Drive.
✓ Repository cloné avec succès dans /content/CADOTProject

📂 Contenu du projet:
ls: cannot access '/content/CADOTProject': No such file or directory

⚠️  ATTENTION: Dossier CADOT_Dataset non trouvé!
   Le dataset n'est peut-être pas dans le repo GitHub.
   Vous devrez l'uploader manuellement ou utiliser Google Drive.


In [3]:
# Monter Google Drive et copier le dataset (uniquement sur Colab)
try:
    from google.colab import drive
    import os
    import shutil
    
    print("📁 Montage de Google Drive...")
    
    # Vérifier si Drive est déjà monté
    if os.path.exists('/content/drive/MyDrive'):
        print("✓ Google Drive déjà monté")
    else:
        try:
            # Essayer le montage avec force_remount=True en cas d'erreur
            drive.mount('/content/drive', force_remount=True)
            print("✓ Google Drive monté avec succès")
        except ValueError as mount_error:
            print(f"❌ Erreur lors du montage de Google Drive: {mount_error}")
            print("\n🔧 SOLUTIONS À ESSAYER:")
            print("1. ⚠️  REDÉMARREZ LE RUNTIME: Runtime > Restart runtime")
            print("   Puis ré-exécutez cette cellule")
            print("\n2. Vérifiez que vous avez autorisé l'accès dans la popup")
            print("\n3. Si le problème persiste:")
            print("   • Déconnectez-vous de Colab et reconnectez-vous")
            print("   • Videz le cache de votre navigateur")
            print("   • Essayez en navigation privée")
            print("\n4. Alternative: Uploadez le dataset directement dans Colab:")
            print("   • Cliquez sur le bouton 'Files' 📁 dans la barre latérale gauche")
            print("   • Créez un dossier 'CADOT_Dataset'")
            print("   • Uploadez vos fichiers dedans")
            print("   (⚠️ Les fichiers seront perdus à la fin de la session)")
            raise
        except Exception as mount_error:
            print(f"❌ Erreur inattendue: {mount_error}")
            print("\n🔧 Essayez de redémarrer le runtime: Runtime > Restart runtime")
            raise
    
    # Chemins possibles du dataset dans Google Drive
    possible_dataset_paths = [
        "/content/drive/MyDrive/CADOT_Dataset",
        "/content/drive/MyDrive/CADOTProject/CADOT_Dataset",
        "/content/drive/My Drive/CADOT_Dataset",
        "/content/drive/My Drive/CADOTProject/CADOT_Dataset",
    ]
    
    dataset_source = None
    for path in possible_dataset_paths:
        if os.path.exists(path):
            dataset_source = path
            print(f"✓ Dataset trouvé dans Google Drive: {path}")
            break
    
    if dataset_source is None:
        print("\n⚠️  Dataset CADOT_Dataset non trouvé dans Google Drive!")
        print("Chemins vérifiés:")
        for path in possible_dataset_paths:
            print(f"  ✗ {path}")
        print("\n📤 UPLOADEZ VOTRE DATASET SUR GOOGLE DRIVE:")
        print("   1. Ouvrez Google Drive dans votre navigateur")
        print("   2. Uploadez le dossier 'CADOT_Dataset'")
        print("   3. Placez-le dans 'Mon Drive/' ou 'Mon Drive/CADOTProject/'")
        print("   4. Attendez la fin de l'upload (peut prendre du temps)")
        print("   5. Ré-exécutez cette cellule")
    else:
        # Copier le dataset dans le projet
        dataset_dest = "/content/CADOTProject/CADOT_Dataset"
        
        if os.path.exists(dataset_dest):
            # Vérifier si le dataset est complet
            train_path = os.path.join(dataset_dest, "train")
            if os.path.exists(train_path):
                import glob
                total_images = len(glob.glob(os.path.join(train_path, "*.jpg"))) + \
                               len(glob.glob(os.path.join(train_path, "*.jpeg"))) + \
                               len(glob.glob(os.path.join(train_path, "*.png")))
                print(f"✓ Dataset déjà présent dans {dataset_dest} ({total_images} images)")
            else:
                print(f"⚠️  Dataset présent mais incomplet, suppression...")
                shutil.rmtree(dataset_dest)
        
        if not os.path.exists(dataset_dest):
            print(f"\n📥 Copie du dataset depuis Google Drive...")
            print(f"   Source: {dataset_source}")
            print(f"   Destination: {dataset_dest}")
            print("   ⏳ Cela peut prendre quelques minutes...")
            
            try:
                shutil.copytree(dataset_source, dataset_dest)
                print(f"✓ Dataset copié avec succès!")
            except Exception as copy_error:
                print(f"❌ Erreur lors de la copie: {copy_error}")
                print("\n💡 Si le dataset est trop gros, utilisez un lien symbolique:")
                print(f"   !ln -s {dataset_source} {dataset_dest}")
                raise
        
        # Vérifier le contenu final
        if os.path.exists(dataset_dest):
            train_path = os.path.join(dataset_dest, "train")
            if os.path.exists(train_path):
                # Compter les images
                import glob
                jpg_files = glob.glob(os.path.join(train_path, "*.jpg"))
                jpeg_files = glob.glob(os.path.join(train_path, "*.jpeg"))
                png_files = glob.glob(os.path.join(train_path, "*.png"))
                total_images = len(jpg_files) + len(jpeg_files) + len(png_files)
                
                print(f"\n📊 Dataset vérifié:")
                print(f"   Dossier train: {train_path}")
                print(f"   Nombre d'images: {total_images}")
                
                if total_images == 0:
                    print(f"\n⚠️  Aucune image trouvée! Vérifiez le contenu du dataset.")
            else:
                print(f"\n⚠️  Dossier train non trouvé dans {dataset_dest}")
    
except ImportError:
    print("Non exécuté sur Colab - Google Drive non monté")

📁 Montage de Google Drive...
❌ Erreur lors du montage de Google Drive: mount failed

🔧 SOLUTIONS À ESSAYER:
1. ⚠️  REDÉMARREZ LE RUNTIME: Runtime > Restart runtime
   Puis ré-exécutez cette cellule

2. Vérifiez que vous avez autorisé l'accès dans la popup

3. Si le problème persiste:
   • Déconnectez-vous de Colab et reconnectez-vous
   • Videz le cache de votre navigateur
   • Essayez en navigation privée

4. Alternative: Uploadez le dataset directement dans Colab:
   • Cliquez sur le bouton 'Files' 📁 dans la barre latérale gauche
   • Créez un dossier 'CADOT_Dataset'
   • Uploadez vos fichiers dedans
   (⚠️ Les fichiers seront perdus à la fin de la session)
❌ Erreur lors du montage de Google Drive: mount failed

🔧 SOLUTIONS À ESSAYER:
1. ⚠️  REDÉMARREZ LE RUNTIME: Runtime > Restart runtime
   Puis ré-exécutez cette cellule

2. Vérifiez que vous avez autorisé l'accès dans la popup

3. Si le problème persiste:
   • Déconnectez-vous de Colab et reconnectez-vous
   • Videz le cache de vo

ValueError: mount failed

In [ ]:
import torch
import os
from pathlib import Path

# Vérification du GPU (CUDA pour Colab, MPS pour Mac)
print(f"PyTorch version: {torch.__version__}")

if torch.cuda.is_available():
    device = "cuda"
    print(f"GPU CUDA détecté: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = "mps"
    print(f"GPU MPS détecté - utilisation de Metal Performance Shaders")
else:
    device = "cpu"
    print("GPU non disponible, utilisation du CPU")

# ========== DÉTECTION AUTOMATIQUE DES CHEMINS ==========
current_dir = Path(os.getcwd())

# Vérifier si on est sur Colab
if "/content" in str(current_dir):
    print("Environnement détecté: Google Colab")
    
    # Chercher le projet dans Google Drive
    possible_paths = [
        Path("/content/drive/MyDrive/CADOTProject"),
        Path("/content/drive/MyDrive/GitHub/CADOTProject"),
        Path("/content/drive/My Drive/CADOTProject"),
        Path("/content/drive/My Drive/GitHub/CADOTProject"),
        Path("/content/CADOTProject"),  # Si uploadé directement dans /content
    ]
    
    project_root = None
    for path in possible_paths:
        if path.exists():
            project_root = path
            print(f"✓ Projet trouvé dans: {project_root}")
            break
    
    if project_root is None:
        print("⚠️  Projet CADOTProject non trouvé dans Google Drive!")
        print("Chemins vérifiés:")
        for path in possible_paths:
            print(f"  ✗ {path}")
        print("\nVeuillez:")
        print("1. Uploader votre dossier CADOTProject sur Google Drive")
        print("2. Ou modifier manuellement la variable project_root ci-dessous")
        # Utiliser un chemin par défaut pour éviter les erreurs
        project_root = Path("/content/drive/MyDrive/CADOTProject")
        print(f"\nChemin par défaut utilisé: {project_root}")
        
else:
    # Environnement local (Mac/Linux/Windows)
    print("Environnement détecté: Local")
    if "LoRA" in str(current_dir):
        # Si on est dans le dossier LoRA, remonter d'un niveau
        project_root = current_dir.parent
    else:
        project_root = current_dir

dataset_root = project_root / "CADOT_Dataset"
train_images = dataset_root / "train"
lora_output = project_root / "LoRA" / "output"

print(f"\nChemins du projet:")
print(f"Racine: {project_root}")
print(f"Dataset: {dataset_root}")
print(f"Images d'entraînement: {train_images}")
print(f"Sortie LoRA: {lora_output}")

# Vérifier que les dossiers existent
if not dataset_root.exists():
    print(f"\n⚠️  ATTENTION: Le dossier dataset n'existe pas: {dataset_root}")
    print(f"   Contenu de {project_root}:")
    if project_root.exists():
        for item in project_root.iterdir():
            print(f"     - {item.name}")
    else:
        print(f"   Le dossier projet n'existe pas non plus!")

if not train_images.exists():
    print(f"\n⚠️  ATTENTION: Le dossier train n'existe pas: {train_images}")
    print(f"   Contenu de {dataset_root}:")
    if dataset_root.exists():
        for item in dataset_root.iterdir():
            print(f"     - {item.name}")

# Créer les dossiers nécessaires
lora_output.mkdir(parents=True, exist_ok=True)
if train_images.exists():
    (train_images / "captions").mkdir(parents=True, exist_ok=True)

print(f"\n✓ Dossiers de sortie créés avec succès")

# Compter les images
if train_images.exists():
    image_files = list(train_images.glob('*.jpg')) + list(train_images.glob('*.jpeg')) + list(train_images.glob('*.png'))
    print(f"  Nombre d'images dans train: {len(image_files)}")
    
    if len(image_files) == 0:
        print(f"\n⚠️  Aucune image trouvée dans {train_images}")
        print(f"   Vérifiez que les images sont bien présentes.")
        print(f"   Fichiers présents dans train:")
        all_files = list(train_images.glob('*'))[:10]  # Afficher max 10 fichiers
        for f in all_files:
            print(f"     - {f.name}")
        if len(list(train_images.glob('*'))) > 10:
            print(f"     ... et {len(list(train_images.glob('*'))) - 10} autres fichiers")
else:
    print(f"\n⚠️  Le dossier train n'existe pas: {train_images}")

PyTorch version: 2.9.0+cu126
GPU non disponible, utilisation du CPU

Chemins du projet:
Racine: /content
Dataset: /content/CADOT_Dataset
Images d'entraînement: /content/CADOT_Dataset/train
Sortie LoRA: /content/LoRA/output

✓ Dossiers créés avec succès
  Nombre d'images dans train: 0


## Préparer les datasets pour l'entraînement

In [14]:
import json
from collections import defaultdict, Counter

# Utiliser les chemins locaux
coco_json_path = train_images / "_annotations.coco.json"
images_dir = train_images
captions_dir = train_images / "captions"

print(f"Lecture du fichier COCO: {coco_json_path}")

# Vérifier si le fichier existe
if not coco_json_path.exists():
    print(f"ERREUR: Fichier COCO introuvable: {coco_json_path}")
    print(f"Fichiers disponibles dans {train_images}:")
    for f in train_images.glob("*"):
        print(f"  {f.name}")
else:
    with open(coco_json_path, "r") as f:
        coco = json.load(f)
    
    print(f"Dataset chargé: {len(coco['images'])} images, {len(coco['annotations'])} annotations")

    images = {img["id"]: img for img in coco["images"]}
    categories = {cat["id"]: cat["name"] for cat in coco["categories"]}
    
    print(f"Catégories disponibles: {list(categories.values())}")

    anns_per_image = defaultdict(list)
    for ann in coco["annotations"]:
        anns_per_image[ann["image_id"]].append(ann)

    def build_caption(anns):
        if not anns:
            return "top-down aerial RGB image of an urban area"
        cls_counts = Counter(categories[a["category_id"]] for a in anns)
        parts = []
        for cls, n in cls_counts.items():
            parts.append(f"{n} {cls}s" if n > 1 else f"one {cls}")
        return "top-down aerial RGB image of an urban area with " + ", ".join(parts)

    # Générer les captions
    caption_count = 0
    for img_id, img_info in images.items():
        file_name = img_info["file_name"]
        anns = anns_per_image.get(img_id, [])
        caption = build_caption(anns)
        txt_path = captions_dir / (Path(file_name).stem + ".txt")
        with txt_path.open("w") as f:
            f.write(caption)
        caption_count += 1
    
    print(f"Captions générées: {caption_count}")

Lecture du fichier COCO: /Users/remyplastre/Documents/GitHub/RoadefChallenge2007/CADOTProject/CADOT_Dataset/train/_annotations.coco.json
Dataset chargé: 3234 images, 75198 annotations
Catégories disponibles: ['small-object', 'basketball field', 'building', 'crosswalk', 'football field', 'graveyard', 'large vehicle', 'medium vehicle', 'playground', 'roundabout', 'ship', 'small vehicle', 'swimming pool', 'tennis court', 'train']
Dataset chargé: 3234 images, 75198 annotations
Catégories disponibles: ['small-object', 'basketball field', 'building', 'crosswalk', 'football field', 'graveyard', 'large vehicle', 'medium vehicle', 'playground', 'roundabout', 'ship', 'small vehicle', 'swimming pool', 'tennis court', 'train']
Captions générées: 3234
Captions générées: 3234


## Charger le modèle TopDown avec Stable Diffusion 1.5

**Note**: Utilisation de Stable Diffusion 1.5 avec les poids LoRA TopDown.

In [13]:
from diffusers import StableDiffusionPipeline
import torch

# Importer peft pour activer le support LoRA
try:
    import peft
    print(f"PEFT {peft.__version__} chargé avec succès")
    # Patch pour forcer diffusers à reconnaître peft
    import diffusers.loaders.lora_pipeline as lora_pipeline_module
    lora_pipeline_module.is_peft_available = lambda: True
except ImportError:
    print("ERREUR: PEFT n'est pas installé. Exécutez: pip install peft")

# Le fichier topdown-sd15.safetensors contient des poids LoRA pour Stable Diffusion 1.5
# Il faut charger un modèle SD 1.5 de base puis appliquer ces poids LoRA dessus

# Pour MPS, utiliser float32 au lieu de float16
dtype = torch.float32 if device == "mps" else (torch.float16 if device == "cuda" else torch.float32)

# Chemin local pour stocker le modèle SD 1.5 (pour éviter de le télécharger à chaque fois)
sd15_local_path = project_root / "models" / "sd15-base"

try:
    # Vérifier si le modèle existe déjà localement
    if sd15_local_path.exists() and (sd15_local_path / "model_index.json").exists():
        print(f"Chargement du modèle SD 1.5 depuis le cache local: {sd15_local_path}")
        pipe = StableDiffusionPipeline.from_pretrained(
            str(sd15_local_path),
            torch_dtype=dtype,
            use_safetensors=True,
            local_files_only=True
        ).to(device)
    else:
        print("Chargement du modèle Stable Diffusion 1.5 depuis le cache HuggingFace...")
        # Charger depuis le cache HuggingFace (pas de téléchargement si déjà en cache)
        pipe = StableDiffusionPipeline.from_pretrained(
            "runwayml/stable-diffusion-v1-5",
            torch_dtype=dtype,
            use_safetensors=True
        ).to(device)
        
        # Sauvegarder le modèle localement pour les prochaines utilisations
        print(f"Sauvegarde du modèle dans: {sd15_local_path}")
        sd15_local_path.mkdir(parents=True, exist_ok=True)
        pipe.save_pretrained(str(sd15_local_path))
        print("Modèle sauvegardé avec succès!")
    
    # Charger les poids LoRA TopDown pour SD 1.5
    topdown_lora_path = project_root / "topdown-sd15.safetensors"
    if topdown_lora_path.exists():
        print(f"Application des poids LoRA TopDown depuis: {topdown_lora_path}")
        try:
            # Méthode alternative: charger manuellement avec safetensors
            from safetensors.torch import load_file
            lora_state_dict = load_file(topdown_lora_path)
            
            # Appliquer les poids LoRA au UNet
            from peft import LoraConfig, inject_adapter_in_model
            from peft.utils import get_peft_model_state_dict
            
            # Configuration LoRA
            lora_config = LoraConfig(
                r=8,  # rang LoRA
                lora_alpha=8,
                target_modules=["to_k", "to_q", "to_v", "to_out.0"],
                lora_dropout=0.0,
            )
            
            # Injecter l'adaptateur LoRA dans le UNet
            pipe.unet = inject_adapter_in_model(lora_config, pipe.unet)
            
            # Charger les poids
            incompatible_keys = pipe.unet.load_state_dict(lora_state_dict, strict=False)
            print(f"LoRA TopDown chargé avec succès!")
            if incompatible_keys.missing_keys:
                print(f"  Clés manquantes: {len(incompatible_keys.missing_keys)}")
            if incompatible_keys.unexpected_keys:
                print(f"  Clés inattendues: {len(incompatible_keys.unexpected_keys)}")
                
        except Exception as lora_error:
            print(f"Impossible de charger le LoRA TopDown: {lora_error}")
            print("Utilisation de SD 1.5 de base sans LoRA pré-entraîné")
    else:
        print("Fichier LoRA TopDown non trouvé, utilisation de SD 1.5 de base")

    # Optimisations mémoire (pas de CPU offload pour MPS)
    if device == "cuda":
        try:
            pipe.enable_model_cpu_offload()
            pipe.enable_attention_slicing()
            print("Optimisations mémoire activées")
        except Exception as opt_error:
            print(f"Optimisations partielles: {opt_error}")
    elif device == "mps":
        try:
            pipe.enable_attention_slicing()
            print("Attention slicing activé pour MPS")
        except Exception as opt_error:
            print(f"Optimisations partielles: {opt_error}")
    
    print(f"Pipeline SD 1.5 + TopDown LoRA chargé avec succès sur {device}")

except Exception as e:
    print(f"Erreur lors du chargement: {e}")
    import traceback
    traceback.print_exc()

PEFT 0.17.1 chargé avec succès
Chargement du modèle SD 1.5 depuis le cache local: /Users/remyplastre/Documents/GitHub/RoadefChallenge2007/CADOTProject/models/sd15-base


Loading pipeline components...: 100%|██████████| 7/7 [00:00<00:00, 16.35it/s]



Application des poids LoRA TopDown depuis: /Users/remyplastre/Documents/GitHub/RoadefChallenge2007/CADOTProject/topdown-sd15.safetensors
LoRA TopDown chargé avec succès!
  Clés manquantes: 942
  Clés inattendues: 792
Attention slicing activé pour MPS
Pipeline SD 1.5 + TopDown LoRA chargé avec succès sur mps
LoRA TopDown chargé avec succès!
  Clés manquantes: 942
  Clés inattendues: 792
Attention slicing activé pour MPS
Pipeline SD 1.5 + TopDown LoRA chargé avec succès sur mps


## Dataset personnalisé pour l'entraînement

In [ ]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import random
from pathlib import Path

class CadotLoraDataset(Dataset):
    def __init__(self, img_dir, cap_dir, size=640):  # 640x640 pour correspondre à la résolution native
        self.img_dir = Path(img_dir)
        self.cap_dir = Path(cap_dir)
        self.size = size
        
        # Chercher les images avec extensions communes
        extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp']
        self.items = []
        for ext in extensions:
            self.items.extend(list(self.img_dir.glob(ext)))
        
        self.items = sorted(self.items)
        print(f"Dataset initialisé avec {len(self.items)} images")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        img_path = self.items[idx]
        txt_path = self.cap_dir / (img_path.stem + ".txt")

        try:
            # Charger et redimensionner l'image
            image = Image.open(img_path).convert("RGB")
            image = image.resize((self.size, self.size), Image.Resampling.LANCZOS)
            
            # Charger la caption
            if txt_path.exists():
                with txt_path.open('r', encoding='utf-8') as f:
                    caption = f.read().strip()
            else:
                caption = "top-down aerial RGB image of an urban area"

            return {"pixel_values": image, "caption": caption}
        
        except Exception as e:
            print(f"Erreur lors du chargement de {img_path}: {e}")
            # Retourner une image par défaut
            image = Image.new('RGB', (self.size, self.size), color='black')
            return {"pixel_values": image, "caption": "top-down aerial RGB image of an urban area"}

# Créer le dataset avec la résolution native 640x640
dataset = CadotLoraDataset(train_images, captions_dir, size=640)
print(f"Dataset créé avec {len(dataset)} échantillons (résolution: 640x640)")

Dataset initialisé avec 3234 images
Dataset créé avec 3234 échantillons


## Configuration des modules LoRA

In [17]:
# Le UNet a déjà le LoRA TopDown appliqué via PEFT
# Nous allons collecter tous les paramètres LoRA pour l'entraînement

unet = pipe.unet
print("Configuration de l'entraînement LoRA...")

# Collecter tous les paramètres entraînables (LoRA)
lora_params = []
for name, param in unet.named_parameters():
    if param.requires_grad:
        lora_params.append(param)

print(f"Paramètres LoRA à entraîner: {len(lora_params)}")

# Vérifier le nombre total de paramètres
total_params = sum(p.numel() for p in lora_params)
print(f"Nombre total de paramètres LoRA: {total_params:,}")

# Afficher quelques exemples de paramètres
print("\nExemples de paramètres LoRA:")
for name, param in list(unet.named_parameters())[:5]:
    if param.requires_grad:
        print(f"  {name}: {param.shape}")

Configuration de l'entraînement LoRA...
Paramètres LoRA à entraîner: 256
Nombre total de paramètres LoRA: 1,594,368

Exemples de paramètres LoRA:


## Boucle d'entraînement optimisée pour GPU local

In [19]:
# ========== OPTIMISATIONS AVANCÉES POUR VITESSE ==========

# 1. Réduire la résolution pour accélérer (sacrifier qualité)
USE_REDUCED_RESOLUTION = True  # Mettre True pour résolution 512x512
RESOLUTION = 512 if USE_REDUCED_RESOLUTION else 640

# 2. Utiliser moins d'epochs pour test rapide
FAST_TRAINING_MODE = True  # Mettre True pour test rapide (1 epoch)
num_epochs = 1 if FAST_TRAINING_MODE else 5

# 3. Utiliser un subset du dataset pour test
USE_SUBSET = True  # Mettre True pour n'utiliser que 10% des images
SUBSET_RATIO = 0.1 if USE_SUBSET else 1.0

# 4. Réduire le nombre d'inference steps pour validation
FAST_INFERENCE = True  # Steps réduits pour génération rapide
INFERENCE_STEPS = 15 if FAST_INFERENCE else 50

# 5. Gradient checkpointing pour économiser la mémoire (ralentit un peu)
USE_GRADIENT_CHECKPOINTING = True  # Recommandé pour 640x640

# 6. Compilation du modèle (PyTorch 2.0+) - EXPÉRIMENTAL sur MPS
USE_TORCH_COMPILE = False  # Mettre True si PyTorch >= 2.0 et MPS compatible

print("╔═══════════════════════════════════════════════════════════╗")
print("║        OPTIMISATIONS DE VITESSE - Configuration         ║")
print("╠═══════════════════════════════════════════════════════════╣")
print(f"║  Résolution:              {RESOLUTION}x{RESOLUTION}                        ║")
print(f"║  Mode rapide:             {'OUI (1 epoch)' if FAST_TRAINING_MODE else 'NON (5 epochs)'}                  ║")
print(f"║  Subset dataset:          {'OUI (10%)' if USE_SUBSET else 'NON (100%)'}                      ║")
print(f"║  Inference rapide:        {'OUI (15 steps)' if FAST_INFERENCE else 'NON (50 steps)'}                 ║")
print(f"║  Gradient checkpointing:  {'ACTIVÉ' if USE_GRADIENT_CHECKPOINTING else 'DÉSACTIVÉ'}                       ║")
print(f"║  Torch compile:           {'ACTIVÉ' if USE_TORCH_COMPILE else 'DÉSACTIVÉ'}                       ║")
print("╚═══════════════════════════════════════════════════════════╝")

# Appliquer gradient checkpointing si activé
if USE_GRADIENT_CHECKPOINTING:
    try:
        pipe.unet.enable_gradient_checkpointing()
        print("✓ Gradient checkpointing activé sur UNet")
    except Exception as e:
        print(f"⚠ Gradient checkpointing non disponible: {e}")

# Compiler le modèle si activé (PyTorch 2.0+)
if USE_TORCH_COMPILE:
    try:
        import torch._dynamo
        torch._dynamo.config.suppress_errors = True
        pipe.unet = torch.compile(pipe.unet, mode="reduce-overhead")
        print("✓ UNet compilé avec torch.compile")
    except Exception as e:
        print(f"⚠ Torch compile non disponible: {e}")

# Recréer le dataset avec la nouvelle résolution si changée
if USE_REDUCED_RESOLUTION or USE_SUBSET:
    dataset = CadotLoraDataset(train_images, captions_dir, size=RESOLUTION)
    
    if USE_SUBSET:
        # Créer un subset pour test rapide
        from torch.utils.data import Subset
        import numpy as np
        subset_size = int(len(dataset) * SUBSET_RATIO)
        indices = np.random.choice(len(dataset), subset_size, replace=False)
        dataset = Subset(dataset, indices)
        print(f"✓ Subset créé: {len(dataset)} images (au lieu de {len(indices)})")
    else:
        print(f"✓ Dataset recréé avec résolution: {RESOLUTION}x{RESOLUTION}")

# Estimation du temps d'entraînement
batches_per_epoch = len(dataset)
total_batches = batches_per_epoch * num_epochs
estimated_time_per_batch = 15 if device == "mps" else 10  # secondes
estimated_total_minutes = (total_batches * estimated_time_per_batch) / 60

print(f"\n📊 Estimation temps d'entraînement:")
print(f"   • Batches par epoch: {batches_per_epoch}")
print(f"   • Epochs: {num_epochs}")
print(f"   • Total batches: {total_batches}")
print(f"   • Temps estimé: ~{estimated_total_minutes:.1f} minutes ({estimated_total_minutes/60:.1f}h)")
print(f"   • Temps par batch: ~{estimated_time_per_batch}s sur {device.upper()}")

╔═══════════════════════════════════════════════════════════╗
║        OPTIMISATIONS DE VITESSE - Configuration         ║
╠═══════════════════════════════════════════════════════════╣
║  Résolution:              512x512                        ║
║  Mode rapide:             OUI (1 epoch)                  ║
║  Subset dataset:          OUI (10%)                      ║
║  Inference rapide:        OUI (15 steps)                 ║
║  Gradient checkpointing:  ACTIVÉ                       ║
║  Torch compile:           DÉSACTIVÉ                       ║
╚═══════════════════════════════════════════════════════════╝
✓ Gradient checkpointing activé sur UNet
Dataset initialisé avec 3234 images
✓ Subset créé: 323 images (au lieu de 323)

📊 Estimation temps d'entraînement:
   • Batches par epoch: 323
   • Epochs: 1
   • Total batches: 323
   • Temps estimé: ~80.8 minutes (1.3h)
   • Temps par batch: ~15s sur MPS


## Optimisations supplémentaires pour accélérer l'entraînement

### Options pour améliorer la vitesse sur Mac Mini M2 16 Go

In [20]:
from torchvision import transforms
from torch.utils.data import DataLoader
import torch
from tqdm.auto import tqdm
import gc
import time

# Configuration des transformations
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

def collate_fn(batch):
    images = [transform(item["pixel_values"]) for item in batch]
    captions = [item["caption"] for item in batch]
    images = torch.stack(images)
    return {"pixel_values": images, "captions": captions}

# ==================== PARAMÈTRES D'ENTRAÎNEMENT - MAC MINI M2 16 GO ====================
batch_size = 1  # Obligatoire pour 640x640 sur M2 16 Go
learning_rate = 1e-4  # Taux optimal pour LoRA fine-tuning
gradient_accumulation_steps = 4  # Simule batch_size=4 pour stabilité
# ====================================================================================

train_loader = DataLoader(
    dataset, 
    batch_size=batch_size, 
    shuffle=True, 
    collate_fn=collate_fn,
    num_workers=0,  # Pas de multiprocessing pour éviter les problèmes MPS
    pin_memory=False  # Désactivé pour MPS
)

# Optimiseur uniquement sur les paramètres LoRA
optimizer = torch.optim.AdamW(lora_params, lr=learning_rate, weight_decay=0.01)

# Learning rate scheduler pour améliorer la convergence
from torch.optim.lr_scheduler import CosineAnnealingLR
scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs * len(train_loader))

# Composants du pipeline
vae = pipe.vae
tokenizer = pipe.tokenizer
text_encoder = pipe.text_encoder
noise_scheduler = pipe.scheduler

print(f"╔══════════════════════════════════════════════════════════╗")
print(f"║  Configuration d'entraînement - Mac Mini M2 16 Go      ║")
print(f"╠══════════════════════════════════════════════════════════╣")
print(f"║  Résolution:        {RESOLUTION}x{RESOLUTION} {'(native)' if RESOLUTION == 640 else '(réduite)'}                    ║")
print(f"║  Epochs:            {num_epochs}                                       ║")
print(f"║  Batch size:        {batch_size}                                       ║")
print(f"║  Gradient accum.:   {gradient_accumulation_steps}                                       ║")
print(f"║  Effective batch:   {batch_size * gradient_accumulation_steps} (simulé)                             ║")
print(f"║  Learning rate:     {learning_rate}                                  ║")
print(f"║  Device:            {device:<35} ║")
print(f"║  Total batches:     {len(train_loader):<35} ║")
print(f"╚══════════════════════════════════════════════════════════╝")

# Mettre les modèles en mode évaluation (sauf UNet)
vae.eval()
text_encoder.eval()

# Timer pour mesurer la vitesse
training_start_time = time.time()

for epoch in range(num_epochs):
    unet.train()
    total_loss = 0
    num_batches = 0
    epoch_start_time = time.time()
    
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    
    for batch_idx, batch in enumerate(progress_bar):
        batch_start_time = time.time()
        
        try:
            with torch.no_grad():
                # Encoder les images
                pixel_values = batch["pixel_values"].to(device)
                # MPS utilise float32, CUDA peut utiliser half
                if device == "cuda":
                    pixel_values = pixel_values.half()
                
                latents = vae.encode(pixel_values).latent_dist.sample()
                latents = latents * vae.config.scaling_factor

                # Encoder le texte
                text_inputs = tokenizer(
                    batch["captions"],
                    padding="max_length",
                    max_length=tokenizer.model_max_length,
                    truncation=True,
                    return_tensors="pt",
                )
                encoder_hidden_states = text_encoder(
                    text_inputs.input_ids.to(device)
                )[0]

            # Ajouter du bruit
            noise = torch.randn_like(latents)
            timesteps = torch.randint(
                0, noise_scheduler.config.num_train_timesteps, 
                (latents.shape[0],), device=device
            ).long()
            noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

            # Prédiction du modèle
            model_pred = unet(
                noisy_latents, 
                timesteps, 
                encoder_hidden_states=encoder_hidden_states
            ).sample

            # Calcul de la perte
            loss = torch.nn.functional.mse_loss(
                model_pred.float(), 
                noise.float()
            )
            
            # Normaliser la perte pour gradient accumulation
            loss = loss / gradient_accumulation_steps

            # Rétropropagation
            loss.backward()
            
            # Mettre à jour les poids seulement après accumulation
            if (batch_idx + 1) % gradient_accumulation_steps == 0:
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

            total_loss += loss.item() * gradient_accumulation_steps
            num_batches += 1
            
            # Calculer la vitesse
            batch_time = time.time() - batch_start_time
            
            # Mise à jour de la barre de progression
            current_lr = scheduler.get_last_lr()[0]
            progress_bar.set_postfix({
                'loss': f'{loss.item() * gradient_accumulation_steps:.4f}',
                'avg_loss': f'{total_loss/num_batches:.4f}',
                'lr': f'{current_lr:.2e}',
                's/batch': f'{batch_time:.1f}'
            })
            
            # Nettoyage mémoire périodique (moins fréquent pour gagner du temps)
            if batch_idx % 20 == 0:
                if device == "cuda":
                    torch.cuda.empty_cache()
                gc.collect()
                
        except Exception as e:
            print(f"Erreur pendant l'entraînement (batch {batch_idx}): {e}")
            continue
    
    # Dernière mise à jour si nécessaire
    if (len(train_loader) % gradient_accumulation_steps) != 0:
        optimizer.step()
        optimizer.zero_grad()
    
    epoch_time = time.time() - epoch_start_time
    avg_loss = total_loss / max(num_batches, 1)
    print(f"✓ Epoch {epoch+1}/{num_epochs} terminé - Perte: {avg_loss:.4f} - Temps: {epoch_time/60:.1f}min ({epoch_time/num_batches:.1f}s/batch)")

total_training_time = time.time() - training_start_time
print(f"\n✓ Entraînement terminé avec succès !")
print(f"  Temps total: {total_training_time/60:.1f} minutes ({total_training_time/3600:.2f}h)")
print(f"  Vitesse moyenne: {total_training_time/num_batches/num_epochs:.1f}s par batch")

╔══════════════════════════════════════════════════════════╗
║  Configuration d'entraînement - Mac Mini M2 16 Go      ║
╠══════════════════════════════════════════════════════════╣
║  Résolution:        512x512 (réduite)                    ║
║  Epochs:            1                                       ║
║  Batch size:        1                                       ║
║  Gradient accum.:   4                                       ║
║  Effective batch:   4 (simulé)                             ║
║  Learning rate:     0.0001                                  ║
║  Device:            mps                                 ║
║  Total batches:     323                                 ║
╚══════════════════════════════════════════════════════════╝


Epoch 1/1: 100%|██████████| 323/323 [10:58:56<00:00, 122.40s/it, loss=0.0641, avg_loss=0.1836, lr=8.56e-05, s/batch=86.8]    



✓ Epoch 1/1 terminé - Perte: 0.1836 - Temps: 658.9min (122.4s/batch)

✓ Entraînement terminé avec succès !
  Temps total: 658.9 minutes (10.98h)
  Vitesse moyenne: 122.4s par batch


## Sauvegarder LoRA après l'entraînement

In [22]:
# Sauvegarder dans le dossier LoRA du dataset
lora_save_path = lora_output / "lora_cadot_topdown"
lora_save_path.mkdir(exist_ok=True)

try:
    # Méthode moderne recommandée avec PEFT
    print("Sauvegarde du modèle LoRA...")
    unet.save_pretrained(str(lora_save_path))
    print(f"✓ LoRA sauvegardé avec succès dans: {lora_save_path}")
    
    # Lister les fichiers sauvegardés
    print("\nFichiers LoRA sauvegardés:")
    for file in lora_save_path.glob("*"):
        size_mb = file.stat().st_size / (1024 * 1024)
        print(f"  • {file.name} ({size_mb:.2f} MB)")
        
except Exception as e:
    print(f"⚠ Erreur avec save_pretrained: {e}")
    
    # Alternative: sauvegarder manuellement les poids LoRA
    try:
        print("\nTentative de sauvegarde manuelle des poids LoRA...")
        from peft import get_peft_model_state_dict
        import torch
        
        lora_state_dict = get_peft_model_state_dict(unet)
        save_file = lora_save_path / "lora_weights.safetensors"
        
        # Sauvegarder avec safetensors pour compatibilité
        from safetensors.torch import save_file as save_safetensors
        save_safetensors(lora_state_dict, str(save_file))
        
        size_mb = save_file.stat().st_size / (1024 * 1024)
        print(f"✓ Poids LoRA sauvegardés manuellement: {save_file.name} ({size_mb:.2f} MB)")
        
    except Exception as e2:
        print(f"❌ Erreur lors de la sauvegarde manuelle: {e2}")
        print("\nConseil: Vérifiez que PEFT est correctement installé avec: pip install peft")

Sauvegarde du modèle LoRA...
✓ LoRA sauvegardé avec succès dans: /Users/remyplastre/Documents/GitHub/RoadefChallenge2007/CADOTProject/CADOT_Dataset/LoRA/lora_cadot_topdown

Fichiers LoRA sauvegardés:
  • config.json (0.00 MB)
  • diffusion_pytorch_model.safetensors (3285.01 MB)
✓ LoRA sauvegardé avec succès dans: /Users/remyplastre/Documents/GitHub/RoadefChallenge2007/CADOTProject/CADOT_Dataset/LoRA/lora_cadot_topdown

Fichiers LoRA sauvegardés:
  • config.json (0.00 MB)
  • diffusion_pytorch_model.safetensors (3285.01 MB)


## Test du modèle LoRA entraîné

In [ ]:
import matplotlib.pyplot as plt
from diffusers import StableDiffusionPipeline

# Recharger le pipeline avec LoRA
try:
    # Utiliser float32 pour MPS
    dtype = torch.float32 if device == "mps" else (torch.float16 if device == "cuda" else torch.float32)
    
    # Utiliser le même pipeline SD 1.5 déjà chargé
    test_pipe = pipe
    
    print("Pipeline de test chargé avec LoRA")

    # Prompts de test
    test_prompts = [
        "top-down aerial RGB image of an industrial area with several trucks and few cars",
        "top-down aerial RGB image of an urban area with many buildings",
        "top-down aerial RGB image of a residential area with houses and streets"
    ]

    # Générer des images de test
    fig, axes = plt.subplots(1, len(test_prompts), figsize=(18, 6))
    if len(test_prompts) == 1:
        axes = [axes]
    
    for i, prompt in enumerate(test_prompts):
        print(f"Génération {i+1}/{len(test_prompts)}: {prompt[:50]}...")
        
        with torch.no_grad():
            image = test_pipe(
                prompt, 
                num_inference_steps=INFERENCE_STEPS,  # Utilise la config d'optimisation
                guidance_scale=7.5,
                height=RESOLUTION,  # Utilise la résolution configurée
                width=RESOLUTION
            ).images[0]
        
        axes[i].imshow(image)
        axes[i].set_title(f"Test {i+1}", fontsize=10)
        axes[i].axis('off')
        
        # Sauvegarder l'image
        output_path = lora_output / f"test_generation_{i+1}_{RESOLUTION}x{RESOLUTION}.png"
        image.save(output_path)
        print(f"✓ Image sauvegardée: {output_path.name}")
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n✓ Test du modèle LoRA terminé avec succès !")
    print(f"  Images générées: {len(test_prompts)}")
    print(f"  Résolution: {RESOLUTION}x{RESOLUTION}")
    print(f"  Inference steps: {INFERENCE_STEPS}")

except Exception as e:
    print(f"Erreur lors du test: {e}")
    import traceback
    traceback.print_exc()

## Nettoyage mémoire

In [ ]:
# Libérer la mémoire GPU
if device == "cuda":
    torch.cuda.empty_cache()
    print("Cache CUDA vidé")
elif device == "mps":
    # MPS n'a pas de cache explicite à vider
    print("Device MPS - pas de cache à vider")

import gc
gc.collect()
print("Nettoyage mémoire terminé")